In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
order_header = spark.read.format("parquet").load("abfss://bronze@intechaccstorage.dfs.core.windows.net/SalesOrderHeader")
order_header.limit(5).display()

SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,CustomerID,ShipToAddressID,BillToAddressID,ShipMethod,CreditCardApprovalCode,SubTotal,TaxAmt,Freight,TotalDue,Comment,rowguid,ModifiedDate,_rescued_data
71774,2,2008-06-01 00:00:00.0000000,2008-06-13 00:00:00.0000000,2008-06-08 00:00:00.0000000,5,False,SO71774,PO348186287,10-4020-000609,29847,1092,1092,CARGO TRANSPORT 5,null,880.3484,70.4279,22.0087,972.7850,null,89e42cdc-8506-48a2-b89b-eb3e64e3554e,2008-06-08 00:00:00.0000000,null
71776,2,2008-06-01 00:00:00.0000000,2008-06-13 00:00:00.0000000,2008-06-08 00:00:00.0000000,5,False,SO71776,PO19952192051,10-4020-000106,30072,640,640,CARGO TRANSPORT 5,null,78.8100,6.3048,1.9703,87.0851,null,8a3448c5-e677-4158-a29b-dd33069be0b0,2008-06-08 00:00:00.0000000,null
71780,2,2008-06-01 00:00:00.0000000,2008-06-13 00:00:00.0000000,2008-06-08 00:00:00.0000000,5,False,SO71780,PO19604173239,10-4020-000340,30113,653,653,CARGO TRANSPORT 5,null,38418.6895,3073.4952,960.4672,42452.6519,null,a47665d2-7ac9-4cf3-8a8b-2a3883554284,2008-06-08 00:00:00.0000000,null
71782,2,2008-06-01 00:00:00.0000000,2008-06-13 00:00:00.0000000,2008-06-08 00:00:00.0000000,5,False,SO71782,PO19372114749,10-4020-000582,29485,1086,1086,CARGO TRANSPORT 5,null,39785.3304,3182.8264,994.6333,43962.7901,null,f1be45a5-5c57-4a50-93c6-5f8be44cb7cb,2008-06-08 00:00:00.0000000,null
71783,2,2008-06-01 00:00:00.0000000,2008-06-13 00:00:00.0000000,2008-06-08 00:00:00.0000000,5,False,SO71783,PO19343113609,10-4020-000024,29957,992,992,CARGO TRANSPORT 5,null,83858.4261,6708.6741,2096.4607,92663.5609,null,7db2329e-6446-42a8-8915-9c8370b68ed8,2008-06-08 00:00:00.0000000,null


In [0]:
order_header = order_header.withColumn("OrderDate", to_date(col("OrderDate")))\
                           .withColumn("DueDate", to_date(col("DueDate")))\
                           .withColumn("ShipDate", to_date(col("ShipDate")))\
                           .withColumn("ModifiedDate", to_date(col("ModifiedDate")))\
                           .withColumn("SubTotal", col("SubTotal").cast('float'))\
                           .withColumn("TaxAmt", col("TaxAmt").cast('float'))\
                           .withColumn("Freight", col("Freight").cast('float'))\
                           .withColumn("TotalDue", col("TotalDue").cast('float'))
                            

In [0]:
order_header = order_header.withColumnRenamed("SalesOrderID", "Sales_order_id")\
                           .withColumnRenamed("RevisionNumber", "Revision_number")\
                           .withColumnRenamed("OrderDate", "Order_date")\
                           .withColumnRenamed("OrderDate", "Order_date")\
                           .withColumnRenamed("DueDate", "Due_date")\
                           .withColumnRenamed("ShipDate", "Ship_date")\
                           .withColumnRenamed("OnlineOrderFlag", "Online_order_flag")\
                           .withColumnRenamed("SalesOrderNumber", "Sales_order_number")\
                           .withColumnRenamed("AccountNumber", "Account_number")\
                           .withColumnRenamed("CustomerID", "Customer_id")\
                           .withColumnRenamed("ShipToAddressID", "Ship_to_address_id")\
                           .withColumnRenamed("BillToAddressID", "Bill_to_address_id")\
                           .withColumnRenamed("ShipMethod", "Ship_method")\
                           .withColumnRenamed("CreditCardApprovalCode", "Credit_card_approval_code")\
                           .withColumnRenamed("SubTotal", "Sub_total")\
                           .withColumnRenamed("TaxAmt", "Tax_amount")\
                           .withColumnRenamed("TotalDue", "Total_due")\
                           .withColumnRenamed("PurchaseOrderNumber", "Purchase_order_number")\
                           .withColumnRenamed("ModifiedDate", "Modified_date")
                

order_header = order_header.drop("Comment", "rowguid", "_rescued_data")
order_header.columns

['Sales_order_id',
 'Revision_number',
 'Order_date',
 'Due_date',
 'Ship_date',
 'Status',
 'Online_order_flag',
 'Sales_order_number',
 'Purchase_order_number',
 'Account_number',
 'Customer_id',
 'Ship_to_address_id',
 'Bill_to_address_id',
 'Ship_method',
 'Credit_card_approval_code',
 'Sub_total',
 'Tax_amount',
 'Freight',
 'Total_due',
 'Modified_date']

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksintech.silver.order_header
(
    Sales_order_id STRING,
    Revision_number STRING,
    Order_date DATE,
    Due_date DATE,
    Ship_date DATE,
    Status STRING,
    Online_order_flag STRING,
    Sales_order_number STRING,
    Purchase_order_number STRING,
    Account_number STRING,
    Customer_id STRING,
    Ship_to_address_id STRING,
    Bill_to_address_id STRING,
    Ship_method STRING,
    Credit_card_approval_code STRING,
    Sub_total FLOAT,
    Tax_amount FLOAT,
    Freight FLOAT,
    Total_due FLOAT,
    Modified_date DATE
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/order_header';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.order_header")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = order_header.alias("source"),
    condition = (
        "target.Sales_order_id = source.Sales_order_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
